# Notebook 02 -- Filtering Pipeline

**Thesis:** Content Moderation on TikTok and X -- A Comparative Analysis of DSA Statements of Reasons (2025)  
**Author:** Mohsen Zahedi  

> **Note:** This notebook is pre-run. The 1,488 original raw parquet files were deleted after filtering to save disk space. Row counts before filtering are taken from the processing log (`02_data_cleaning/logs/data_profile.md`). The current working files at `C:\BA_Data\` reflect the post-filtering state.

---

This notebook documents the three filtering steps applied to the raw DSA data before analysis. Each step narrows the dataset to the records relevant for this thesis.

In [1]:
from pathlib import Path

import polars as pl

DATA_ROOT = Path(r"C:\BA_Data")
TIKTOK_FILE = DATA_ROOT / "tiktok_de_2025.parquet"
X_FILE = DATA_ROOT / "x_de_2025.parquet"

## Step 1 -- Territorial Scope Filter (DE)

The DSA Transparency Database contains SoRs for all EEA countries. This thesis focuses on **Germany (DE)** only.

The filter kept any record where the `territorial_scope` field contains `DE` -- this includes both single-country (`DE`) and multi-country strings (`DE,FR,NL,IT,...`). Multi-country records represent moderation decisions that apply EEA-wide and happen to cover Germany.

**Script:** `02_data_cleaning/scripts/filter_split_save_de_polars.py`

| Platform | Rows before DE filter | Rows after DE filter | Retention |
|---|---|---|---|
| TikTok | 1,002,729,268 | 999,079,277 | 99.6% |
| X | 670,093 | 183,324 | 27.4% |

**Key finding:** TikTok retained almost all records because it enforces content moderation EEA-wide (most records include DE). X retained only 27.4% because it enforces country-by-country -- the majority of its records target other EEA countries.

In [2]:
# Show what territorial_scope values look like after the DE filter
# TikTok: multi-country strings containing DE
print("TikTok -- sample territorial_scope values:")
(
    pl.scan_parquet(str(TIKTOK_FILE))
    .select("territorial_scope")
    .head(8)
    .collect()
    .to_series()
    .to_list()
)


TikTok -- sample territorial_scope values:


['["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","

In [3]:
# X: single country code
print("X -- unique territorial_scope values:")
(
    pl.scan_parquet(str(X_FILE))
    .select("territorial_scope")
    .unique()
    .collect()
    .to_series()
    .to_list()
)

X -- unique territorial_scope values:


['["DE"]']

## Step 2 -- Date Range Filter (2025)

Some records in the DE-filtered files had `application_date` values outside the 2025 calendar year. These were dropped to keep the dataset within the thesis scope.

**Script:** `02_data_cleaning/scripts/apply_date_filter.py`

| Platform | Rows before date filter | Rows dropped | Rows after date filter |
|---|---|---|---|
| TikTok | 999,079,277 | 1,391,010 | 997,688,267 |
| X | 183,324 | 3 | 183,321 |

In [4]:
# Confirm the date range of the current working files
for label, file in [("TikTok", TIKTOK_FILE), ("X", X_FILE)]:
    result = (
        pl.scan_parquet(str(file))
        .select(
            pl.col("application_date").min().alias("earliest"),
            pl.col("application_date").max().alias("latest"),
        )
        .collect()
    )
    print(f"{label}: {result['earliest'][0]} to {result['latest'][0]}")

TikTok: 2025-01-01 00:00:00 to 2025-12-12 00:00:00
X: 2025-01-01 00:00:00 to 2025-12-31 00:00:00


## Step 3 -- API Version Split

On **July 1, 2025** the DSA Commission released API v2, which changed the `category` vocabulary. Records submitted before this date used v1 labels; records from July 1 onwards use v2 labels. The `api_version` column captures this distinction.

This split matters because v1 and v2 category labels are not directly comparable -- categories were renamed, removed, and introduced. Harmonization is required before any cross-period analysis. See Notebook 03.

In [5]:
# API version distribution per platform
for label, file in [("TikTok", TIKTOK_FILE), ("X", X_FILE)]:
    dist = (
        pl.scan_parquet(str(file))
        .group_by("api_version")
        .agg(pl.len().alias("count"))
        .sort("api_version")
        .collect()
    )
    total = dist["count"].sum()
    print(f"\n{label} ({total:,} total):")
    for row in dist.iter_rows(named=True):
        pct = row["count"] / total * 100
        print(f"  {row['api_version']}: {row['count']:>13,}  ({pct:.1f}%)")


TikTok (997,688,267 total):
  v1:   830,222,338  (83.2%)
  v2:   167,465,929  (16.8%)

X (183,321 total):
  v1:       132,191  (72.1%)
  v2:        51,130  (27.9%)


## Summary -- Filtering Results

| Step | TikTok rows | X rows |
|---|---|---|
| Raw (before any filtering) | 1,002,729,268 | 670,093 |
| After DE territorial filter | 999,079,277 | 183,324 |
| After 2025 date filter | 997,688,267 | 183,321 |
| -- of which v1 | 830,222,338 | 132,191 |
| -- of which v2 | 167,465,929 | 51,130 |

The working files at `C:\BA_Data\tiktok_de_2025.parquet` and `C:\BA_Data\x_de_2025.parquet` represent the post-filtering state. These are the inputs for the harmonization step documented in Notebook 03.